# Diseño de Redes con Algoritmos Genéticos

Cuaderno listo para **Google Colab** o **JupyterLab**.

Incluye:
- representación binaria de enlaces,
- construcción del grafo,
- función de fitness con penalización,
- reparación de redes no conexas,
- gráfico de convergencia,
- visualización de la topología final.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx


## Datos del problema


In [ ]:
n_nodos = 6
enlaces = [
    (0, 1, 4),
    (0, 2, 3),
    (0, 3, 7),
    (1, 2, 6),
    (1, 4, 5),
    (2, 3, 4),
    (2, 4, 2),
    (2, 5, 8),
    (3, 5, 3),
    (4, 5, 6)
]
n_enlaces = len(enlaces)

tam_poblacion = 100
n_generaciones = 120
prob_cruce = 0.8
prob_mutacion = 0.05
tam_torneo = 3

random.seed(42)
np.random.seed(42)


## Funciones auxiliares


In [ ]:
def crear_individuo():
    return np.random.randint(0, 2, size=n_enlaces)

def construir_grafo(individuo):
    G = nx.Graph()
    G.add_nodes_from(range(n_nodos))
    for gen, (u, v, costo) in zip(individuo, enlaces):
        if gen == 1:
            G.add_edge(u, v, weight=costo)
    return G

def costo_total(individuo):
    return sum(costo for gen, (_, _, costo) in zip(individuo, enlaces) if gen == 1)

def cantidad_componentes(individuo):
    G = construir_grafo(individuo)
    return nx.number_connected_components(G)

def fitness(individuo):
    costo = costo_total(individuo)
    componentes = cantidad_componentes(individuo)
    if componentes == 1:
        return -costo
    penalizacion = 100 * (componentes - 1)
    return -(costo + penalizacion)

def reparar(individuo):
    individuo = individuo.copy()
    G = construir_grafo(individuo)
    while not nx.is_connected(G):
        componentes = list(nx.connected_components(G))
        comp_a = list(componentes[0])
        comp_b = list(componentes[1])
        posibles = []
        for i, (u, v, costo) in enumerate(enlaces):
            if individuo[i] == 0:
                if (u in comp_a and v in comp_b) or (u in comp_b and v in comp_a):
                    posibles.append((i, costo))
        if posibles:
            mejor_idx = min(posibles, key=lambda x: x[1])[0]
            individuo[mejor_idx] = 1
        else:
            idx = random.randint(0, n_enlaces - 1)
            individuo[idx] = 1
        G = construir_grafo(individuo)
    return individuo

def seleccion_torneo(poblacion):
    candidatos = random.sample(poblacion, tam_torneo)
    return max(candidatos, key=fitness).copy()

def cruce_un_punto(padre1, padre2):
    if random.random() < prob_cruce:
        punto = random.randint(1, n_enlaces - 1)
        hijo1 = np.concatenate([padre1[:punto], padre2[punto:]])
        hijo2 = np.concatenate([padre2[:punto], padre1[punto:]])
        return hijo1, hijo2
    return padre1.copy(), padre2.copy()

def mutar(individuo):
    mutado = individuo.copy()
    for i in range(len(mutado)):
        if random.random() < prob_mutacion:
            mutado[i] = 1 - mutado[i]
    return mutado


## Ejecución del algoritmo genético


In [ ]:
poblacion = [crear_individuo() for _ in range(tam_poblacion)]
poblacion = [reparar(ind) for ind in poblacion]

mejores_costos = []
promedios_costos = []
mejor_individuo_global = None
mejor_fitness_global = -float('inf')

for generacion in range(n_generaciones):
    nueva_poblacion = []
    while len(nueva_poblacion) < tam_poblacion:
        padre1 = seleccion_torneo(poblacion)
        padre2 = seleccion_torneo(poblacion)
        hijo1, hijo2 = cruce_un_punto(padre1, padre2)
        hijo1 = mutar(hijo1)
        hijo2 = mutar(hijo2)
        hijo1 = reparar(hijo1)
        hijo2 = reparar(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < tam_poblacion:
            nueva_poblacion.append(hijo2)
    poblacion = nueva_poblacion
    fitness_poblacion = [fitness(ind) for ind in poblacion]
    mejor_generacion = max(poblacion, key=fitness)
    mejor_fit = fitness(mejor_generacion)
    promedio_fit = np.mean(fitness_poblacion)
    mejores_costos.append(-mejor_fit)
    promedios_costos.append(-promedio_fit)
    if mejor_fit > mejor_fitness_global:
        mejor_fitness_global = mejor_fit
        mejor_individuo_global = mejor_generacion.copy()

mejor_individuo_global = reparar(mejor_individuo_global)
costo_final = costo_total(mejor_individuo_global)
G_final = construir_grafo(mejor_individuo_global)

print('Mejor solución encontrada:', mejor_individuo_global.tolist())
print('Costo total:', costo_final)
print('¿La red es conexa?:', nx.is_connected(G_final))
print('Enlaces seleccionados:')
for gen, (u, v, costo) in zip(mejor_individuo_global, enlaces):
    if gen == 1:
        print(f'{u} - {v}  costo={costo}')


## Gráfico de convergencia


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_costos, label='Mejor costo')
plt.plot(promedios_costos, label='Costo promedio')
plt.xlabel('Generación')
plt.ylabel('Costo')
plt.title('Evolución del algoritmo genético')
plt.legend()
plt.grid(True)
plt.show()


## Visualización de la red final


In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G_final, seed=42)
nx.draw(G_final, pos, with_labels=True, node_size=800, font_size=10)
labels = nx.get_edge_attributes(G_final, 'weight')
nx.draw_networkx_edge_labels(G_final, pos, edge_labels=labels)
plt.title('Topología de la mejor red encontrada')
plt.show()


## Costos de enlaces seleccionados


In [ ]:
etiquetas = []
costos = []
for gen, (u, v, costo) in zip(mejor_individuo_global, enlaces):
    if gen == 1:
        etiquetas.append(f'{u}-{v}')
        costos.append(costo)

plt.figure(figsize=(8, 5))
plt.bar(etiquetas, costos)
plt.xlabel('Enlaces')
plt.ylabel('Costo')
plt.title('Costos de los enlaces seleccionados')
plt.grid(axis='y')
plt.show()


## Ejercicio sugerido

Modificar:
- la cantidad de nodos,
- los costos de enlaces,
- la probabilidad de mutación,
- la penalización por desconexión,

y analizar cómo cambia la topología final obtenida.
